# Archived team notebook
See [workflow and limitations](README.md) before running. Outputs and session metadata have been cleared. Supply your own data paths and checkpoints. These notebooks were not rerun during packaging.

In [ ]:
# 安裝必要套件（只需執行一次）
!pip install open-clip-torch ftfy regex tqdm -q

In [ ]:
# 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# STEP 3：設定路徑與載入 prompt 候選
import os

image_dir = "/content/drive/MyDrive/lora_crack_dataset/Crack_Dataset_512"
prompt_file = "/content/drive/MyDrive/lora_crack_dataset/prompt_candidates.txt"

output_dir = "/content/lora_selected"
os.makedirs(output_dir, exist_ok=True)

with open(prompt_file, "r") as f:
    candidates = [line.strip() for line in f if line.strip()]
print(f"載入 prompt 候選：{len(candidates)} 條")

In [ ]:
# STEP 4：載入模型
import torch
import open_clip
from PIL import Image
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
model = model.to(device).eval()
tokenizer = open_clip.get_tokenizer('ViT-L-14')

In [ ]:
# STEP 5：將所有 prompt 編碼成向量
with torch.no_grad():
    text_tokens = tokenizer(candidates).to(device)
    text_features = model.encode_text(text_tokens)
    text_features /= text_features.norm(dim=-1, keepdim=True)

In [ ]:

# STEP 6：逐張圖片選出最佳 prompt
output_prompts = []

for fname in sorted(os.listdir(image_dir)):
    if not fname.lower().endswith(".jpg"): continue

    image_path = os.path.join(image_dir, fname)
    image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        similarity = image_features @ text_features.T
        best_idx = similarity.argmax().item()
        best_prompt = candidates[best_idx]

    # 寫出 .txt 描述
    with open(os.path.join(output_dir, os.path.splitext(fname)[0] + ".txt"), "w") as f:
        f.write(best_prompt)

    # 複製圖像
    from shutil import copy2
    copy2(image_path, os.path.join(output_dir, fname))

    output_prompts.append(f"{fname} → {best_prompt}")
    print(f"✅ {fname} → {best_prompt}")

In [ ]:
# STEP 7：儲存 prompt 對應表
with open(os.path.join(output_dir, "prompt_mapping.txt"), "w") as f:
    f.write("\n".join(output_prompts))

In [ ]:
# STEP 8：打包為 zip
from zipfile import ZipFile

zip_path = "/content/drive/MyDrive/lora_crack_dataset/lora_crack_dataset.zip" ###
with ZipFile(zip_path, 'w') as zipf:
    for fname in sorted(os.listdir(output_dir)):
        full_path = os.path.join(output_dir, fname)
        zipf.write(full_path, arcname=fname)

print(f"\n 已完成並打包：{zip_path}")